# Where bars show

The loop only reports progress. What shows it is chosen by where the code runs: a widget
in a notebook that has anywidget, plain HTML in one that does not, a line that is redrawn
on a terminal, and a line every few seconds in a pipe, a file or a CI job.

In [ ]:
import sys
import time

from anyprogressbar import LogRenderer, progress, set_renderer, write

## Choosing by name

`set_renderer` takes `"notebook"`, `"widget"`, `"html"`, `"terminal"`, `"log"` or `"off"`.
The `ANYPROGRESSBAR` environment variable takes the same names, for jobs you do not want
to change. This is the bar without the widget:

In [ ]:
set_renderer("html")
for _ in progress(range(40), "Plain HTML"):
    time.sleep(0.05)

And nothing at all:

In [ ]:
set_renderer("off")
for _ in progress(range(40), "Silent"):
    time.sleep(0.01)

## Log lines

What a pipe or a CI job gets: lines that are only ever added. By default one every five
seconds, on standard error, and none for work done within two seconds; here they come
faster, to show them.

In [ ]:
set_renderer(LogRenderer(sys.stdout, interval=0.5, delay=0))
for _ in progress(range(40), "Nightly import", unit="files"):
    time.sleep(0.05)

`None` goes back to choosing by where the code runs.

In [ ]:
set_renderer(None)

## One bar, another renderer

A single bar can be given a renderer of its own, or be switched off (`disable=True`, which
costs nothing in the loop).

In [ ]:
lines = LogRenderer(sys.stdout, interval=1, delay=0)
for _ in progress(range(40), "Only this one", renderer=lines):
    time.sleep(0.03)

for _ in progress(range(40), "Not shown", disable=True):
    pass

## Printing while a bar runs

Print and log as usual. On a terminal, while bars are live, what goes to standard output,
standard error or a logging handler that writes to them appears above the bars, and the
bars are drawn again under it. A prompt (`input("Name: ")`) has the screen to itself until
it is answered. In a notebook a bar is an output of its own, so printing never touched it.

In [ ]:
import logging

logging.basicConfig(format="%(levelname)s %(message)s", stream=sys.stdout, force=True)

for i in progress(range(30), "Checking"):
    time.sleep(0.05)
    if i % 10 == 9:
        print(f"{i + 1} checked")
    if i == 20:
        logging.warning("one file was skipped")

`write` does the same for a stream that is not redirected, such as a file of your own
(here one in memory):

In [ ]:
import io

report = io.StringIO()
for i in progress(range(20), "Writing a report"):
    time.sleep(0.05)
    write(f"line {i}", file=report)
print(report.getvalue().splitlines()[-1])

## A renderer of your own

A renderer is any object with `update(root)` and `close(root)`. It is given a `Snapshot`:
plain data about the bar and its children at one moment.

In [ ]:
class Milestones:
    """Says when each quarter is passed."""

    def __init__(self):
        self.said = 0

    def update(self, root):
        quarter = int(4 * (root.fraction or 0))
        if quarter > self.said:
            self.said = quarter
            print(f"{root.desc}: {25 * quarter}% after {root.elapsed:.1f} s")

    def close(self, root):
        print(f"{root.desc}: {root.status} ({root.n:.0f} of {root.total:.0f})")

for _ in progress(range(40), "Milestones", renderer=Milestones()):
    time.sleep(0.03)